In [2]:
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import re
import requests
from datetime import datetime

# API Testing for additional data

API rate limits take into account client identity to determine the level of access.
Stronger forms of identification result in a higher limit, such as running in Wikimedia Cloud Services (WMCS) or authenticating requests.
The highest limits require running in WMCS, community bot approval, or being well-known to the Wikimedia Foundation.
Further limits or additional best practices may apply to specific APIs.

In [13]:
# Wikipedia bot with a compliant user-agent

# Wikipedia bot/user-agent policy requires identifying the bot clearly and providing contact info.
# Use the official MediaWiki API instead of scraping page HTML when possible.
headers = {
    "User-Agent": "CSS_ProjectBot/1.0 (https://github.com/Stampe04/CSS_Project; fwaggot.player@gmail.com)",
    "From": "fwaggot.player@gmail.com",
    "Accept": "application/json"
}

url = "https://en.wikipedia.org/w/api.php"
params = {
    "action": "query",
    "list": "allusers",
    "augroup": "sysop",
    "aulimit": "max",
    "format": "json"
}

admins = []

while True:
    response = requests.get(url, headers=headers, params=params, timeout=10)
    response.raise_for_status()
    data = response.json()

    admins.extend([user["name"] for user in data["query"]["allusers"]])

    if "continue" not in data:
        break

    params.update(data["continue"])

print(f"Administrators: {len(admins)}")
for admin in admins[::10]: # print every 10th admind
    print("-", admin)


# save the list of admins to a CSV file for later analysis
admins_df = pd.DataFrame({
    "Admin": admins,
    "Active": None,
    "Semi-Active": None,
    "Inactive": None
})
admins_df.to_csv("wikipedia_admins.csv", index=False)
print("Saved list of administrators to wikipedia_admins.csv")

Administrators: 811
- 28bytes
- Acalamari
- Ajpolino
- Amorymeltzer
- Antandrus
- Atlant
- BDD
- BethNaught
- Black Kite
- Brian Kendig
- CJCurrie
- Canadian Paul
- Cburnett
- Chris G
- Colin M
- Cryptic
- DYKUpdateBot
- DatGuy
- Deb
- DerHexer
- Doczilla
- Drmies
- Edcolins
- EncMstr
- Extraordinary Writ
- Feezo
- Frank
- Galobtter
- Gentgeen
- GoldRingChip
- Ground Zero
- Harrias
- Hiding
- Ianblair23
- Isabelle Belato
- JIP
- Jamesofur
- Jesse Viviano
- John K
- K6ka
- Keilana
- KrakatoaKatie
- LFaraone
- Lexicon
- Lustiger seth
- Mailer diablo
- Mattythewhite
- Metamagician3000
- Mike Rosoft
- Mjroots
- Morwen
- NativeForeigner
- NinjaRobotPirate
- Ohnoitsjamie
- PFHLai
- PedanticallySpeaking
- Pinkville
- Prolog
- Ragesoss
- RegentsPark
- Rjjiii
- Rogerd
- SQL
- SarekOfVulcan
- Sdrqaz
- Sesel
- SimonP
- Smartse
- Spellcast
- Steven Walling
- TadejM
- The Anome
- The ed17
- TigerShark
- Tom Morris
- U193581
- Valfontis
- Visviva
- Wehwalt
- Wugapodes
- Yaris678
- Zzyzx11
Saved list

# Data Collection for each admin

Now we need to fetch structured metrics for multiple admins.

In [14]:
# Example: fetch structured metrics for multiple admins using the MediaWiki API
# This is more reliable than scraping the HTML page, because the API returns fields directly.

if admins:
    target_admins = admins[:5]  # Fetch metrics for the first 5 admins instead of just one
else:
    target_admins = ["Example"]

params = {
    "action": "query",
    "list": "users",
    "ususers": "|".join(target_admins),
    "usprop": "editcount|groups|registration|rights|blockinfo",
    "format": "json"
}

response = requests.get(url, headers=headers, params=params, timeout=10)
response.raise_for_status()

user_data = response.json()
users = user_data["query"]["users"]

for user in users:
    print(f"\nAdmin metrics for: {user.get('name')}")
    print("editcount:", user.get("editcount"))
    print("registration:", user.get("registration"))
    print("groups:", user.get("groups"))
    print("rights:", user.get("rights"))
    print("blocked:", user.get("blocked"))


Admin metrics for: 28bytes
editcount: 32813
registration: 2006-12-14T23:17:44Z
groups: ['autoreviewer', 'bureaucrat', 'sysop', '*', 'user', 'autoconfirmed']
rights: ['autopatrol', 'move-subpages', 'suppressredirect', 'tboverride', 'noratelimit', 'checkuser-temporary-account', 'checkuser-temporary-account-auto-reveal', 'oathauth-verify-user', 'oathauth-view-log', 'override-antispoof', 'templateeditor', 'changetags', 'extendedconfirmed', 'deleterevision', 'deletelogentry', 'editcontentmodel', 'block', 'createaccount', 'delete', 'deletedhistory', 'deletedtext', 'undelete', 'editinterface', 'editsitejson', 'edituserjson', 'import', 'move', 'move-rootuserpages', 'move-categorypages', 'patrol', 'protect', 'editprotected', 'rollback', 'upload', 'reupload', 'reupload-shared', 'unwatchedpages', 'autoconfirmed', 'editsemiprotected', 'ipblock-exempt', 'blockemail', 'markbotedits', 'apihighlimits', 'browsearchive', 'movefile', 'mergehistory', 'managechangetags', 'deletechangetags', 'abusefilter-r

# Wikipedia_admins Update

Extract admin names from each category ```Active```, ```Semi-Active```, ```Inactive``` and then update the admin list to further develop the communities (WIP).

In [15]:
from bs4 import BeautifulSoup

# Extract admin names from each activity category by fetching separate pages
activity_categories = {
    "Active": set(),
    "Semi-Active": set(),
    "Inactive": set()
}

# Map of category names to URL suffixes
category_pages = {
    "Active": "Active",
    "Semi-Active": "Semi-active",
    "Inactive": "Inactive"
}

# Fetch each category page separately
for category, suffix in category_pages.items():
    admin_list_params = {
        "action": "parse",
        "page": f"Wikipedia:List_of_administrators/{suffix}",
        "format": "json"
    }
    
    try:
        response = requests.get(url, headers=headers, params=admin_list_params, timeout=10)
        response.raise_for_status()
        page_data = response.json()
        
        # Get the HTML content
        html_content = page_data["parse"]["text"]["*"]
        
        print(f"\nFetching: Wikipedia:List_of_administrators/{suffix}")
        
        # Use regex to find all user links directly from HTML
        # Pattern: href="/wiki/User:USERNAME" or href="/wiki/User_talk:USERNAME"
        # Extract the username part
        import re
        user_links = re.findall(r'href="/wiki/User(?:_talk)?:([^"]+)"', html_content)
        
        for username in user_links:
            # Clean up the username (remove URL encoding, etc.)
            username = username.replace("_", " ").strip()
            if username and username.lower() not in ["edit", "history", "talk", "contribs", "logs", "view logs"]:
                activity_categories[category].add(username)
        
        print(f"  -> Extracted {len(activity_categories[category])} admins")
        # Check if "5 albert square" is in this category
        # if "5 albert square" in activity_categories[category]:
        #     print(f"     ✓ '5 albert square' found!")
        # # Show first 10 for verification
        # sample = sorted(list(activity_categories[category]))[:10]
        # print(f"     Sample: {sample}")
    
    except Exception as e:
        print(f"  Error fetching {suffix}: {e}")

# Cross-reference admins with activity categories
admins_df["Active"] = admins_df["Admin"].isin(activity_categories["Active"]).astype(int)
admins_df["Semi-Active"] = admins_df["Admin"].isin(activity_categories["Semi-Active"]).astype(int)
admins_df["Inactive"] = admins_df["Admin"].isin(activity_categories["Inactive"]).astype(int)

# Save updated CSV
admins_df.to_csv("wikipedia_admins.csv", index=False)
print("\nUpdated CSV saved to wikipedia_admins.csv")


Fetching: Wikipedia:List_of_administrators/Active
  -> Extracted 414 admins

Fetching: Wikipedia:List_of_administrators/Semi-active
  -> Extracted 317 admins

Fetching: Wikipedia:List_of_administrators/Inactive
  -> Extracted 79 admins

Updated CSV saved to wikipedia_admins.csv


In [16]:
# Debug: Show extracted usernames from each category
print("Sample extracted usernames:")
for category, usernames in activity_categories.items():
    sample = list(usernames)[:5]
    print(f"{category}: {sample}")

print("\n" + "="*50)
# Find unmatched Active admins
unmatched_active = [u for u in activity_categories['Active'] if u not in admins_df['Admin'].values]
print(f"\nUnmatched Active admins ({len(unmatched_active)} total):")
for admin in unmatched_active[:10]:
    print(f"  - '{admin}'")
    # Look for similar names in admins_df
    similar = [a for a in admins_df['Admin'] if admin.lower() == a.lower()]
    if similar:
        print(f"    Found match with different case: {similar}")

Sample extracted usernames:
Active: ['Wehwalt', 'Morwen', 'Z1720', 'Rjjiii', 'Dave souza']
Semi-Active: ['Keilana', 'Red-tailed hawk', 'Ceranthor', 'The JPS', 'TunnelESON']
Inactive: ['Smith609', 'Calliopejen1', 'Nlu', 'Davidcannon', 'SQL']


Unmatched Active admins (3 total):
  - 'Yamaguchi%E5%85%88%E7%94%9F'
  - 'HickoryOughtShirt%3F4'
  - 'R%27n%27B'


# Update all community statistics

Using the new metrics, we update our list and create new statistics and communities that better explain each community and possibly discover underlying metrics that define a community (WIP).

In [6]:
# Load admins from the existing CSV file instead of using the admins list from earlier
admins_df = pd.read_csv("wikipedia_admins.csv")
admins = admins_df["Admin"].tolist()

# Example: fetch structured metrics for multiple admins using the MediaWiki API
# This is more reliable than scraping the HTML page, because the API returns fields directly

headers = {
    "User-Agent": "CSS_ProjectBot/1.0 (https://github.com/Stampe04/CSS_Project; fwaggot.player@gmail.com)",
    "From": "fwaggot.player@gmail.com",
    "Accept": "application/json"
}

url = "https://en.wikipedia.org/w/api.php"
params = {
    "action": "query",
    "list": "allusers",
    "augroup": "sysop",
    "aulimit": "max",
    "format": "json"
}

# Initialize new columns if they don't exist
if "editcount" not in admins_df.columns:
    admins_df["editcount"] = None
if "registration_years" not in admins_df.columns:
    admins_df["registration_years"] = None

# Ensure registration_years is object dtype to store string values
admins_df["registration_years"] = admins_df["registration_years"].astype('object')

# Batch admins into groups of 50 (API limit)
batch_size = 50
all_users = []

for i in range(0, len(admins), batch_size):
    batch = admins[i:i+batch_size]
    
    params = {
        "action": "query",
        "list": "users",
        "ususers": "|".join(batch),
        "usprop": "editcount|registration",
        "format": "json"
    }
    
    try:
        # Use POST instead of GET to avoid URI too long errors
        response = requests.post(url, headers=headers, data=params, timeout=10)
        response.raise_for_status()
        user_data = response.json()
        users = user_data["query"]["users"]
        all_users.extend(users)
        print(f"Fetched batch {i//batch_size + 1}: {len(users)} users")
    except Exception as e:
        print(f"Error fetching batch {i//batch_size + 1}: {e}")

# Update the dataframe with fetched data
for user in all_users:
    username = user.get("name")
    editcount = user.get("editcount")
    registration_years = user.get("registration")
    
    mask = admins_df["Admin"] == username
    if mask.any():
        admins_df.loc[mask, "editcount"] = editcount
        admins_df.loc[mask, "registration_years"] = registration_years

# Save updated CSV
admins_df.to_csv("wikipedia_admins.csv", index=False)
print(f"\nUpdated CSV saved to wikipedia_admins.csv with {len(all_users)} admin records")

Fetched batch 1: 50 users
Fetched batch 2: 50 users
Fetched batch 3: 50 users
Fetched batch 4: 50 users
Fetched batch 5: 50 users
Fetched batch 6: 50 users
Fetched batch 7: 50 users
Fetched batch 8: 50 users
Fetched batch 9: 50 users
Fetched batch 10: 50 users
Fetched batch 11: 50 users
Fetched batch 12: 50 users
Fetched batch 13: 50 users
Fetched batch 14: 50 users
Fetched batch 15: 50 users
Fetched batch 16: 50 users
Fetched batch 17: 11 users

Updated CSV saved to wikipedia_admins.csv with 811 admin records
